# Explore the processed-layer inputs

Loads every `data/input/latest_*.parquet` into a DataFrame so you can poke at it.

**Kernel: `Python 3.14 (ftm2j jobs)`** — the `jobs/.venv` interpreter, which is the one
with pandas and pyarrow. The repo-root `.venv` has neither and cannot run this. If the
kernel is missing from the picker, re-register it:

```bash
jobs/.venv/bin/python -m ipykernel install --user --name ftm2j-jobs --display-name "Python 3.14 (ftm2j jobs)"
```

Or skip the picker entirely and launch Lab from the venv itself:

```bash
cd jobs && uv run --group dev jupyter lab explore_inputs.ipynb
```

`data/` is gitignored — these are local artifacts synced out of S3, not fixtures.
What the columns mean, and which ones `build_dataset.py` actually uses, is in
[README.md](README.md).

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# Walk up rather than assuming cwd is jobs/. A notebook has no __file__, and cwd depends
# on how the kernel was launched — an editor may start it at the repo root instead.
INPUT_DIR = next(p / "data" / "input" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "input").is_dir())

paths = {p.name.removeprefix("latest_").removesuffix(".parquet"): p for p in sorted(INPUT_DIR.glob("latest_*.parquet"))}
paths

## What's in the files before loading them

Row counts and sizes first — `shareholders` is 2.2M rows / ~130 MB on disk, so it is
worth knowing what you're about to pull into memory.

In [ ]:
pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": pq.ParquetFile(path).metadata.num_rows,
            "columns": pq.ParquetFile(path).metadata.num_columns,
            "mb_on_disk": round(path.stat().st_size / 1e6, 1),
        }
        for name, path in paths.items()
    ]
).set_index("dataset")

## Load

Everything lands in `frames`, and each dataset also gets a bare name —
`company_info`, `corporate_structure`, `shareholders`, `cdt`, `cdt_items`,
`cdt_mentions`.

CDT is three files, not one. `cdt` (debt-instruments) holds the instruments and
carries no provenance at all — no url, filing date, or accession — so a citation
only exists by joining it through `cdt_mentions` to `cdt_items`. `cdt_mentions` is
also the only place the currency lives, in `amount_json`.

`corporate_structure` comes back one column short of the count above: its
`__index_level_0__` is a saved pandas index and becomes the DataFrame's index, not a
column.

To skip the big one while iterating, pass `columns=[...]` to `read_parquet`, or drop
it from `paths` above.

In [ ]:
frames = {name: pd.read_parquet(path) for name, path in paths.items()}
globals().update(frames)

for name, df in frames.items():
    print(f"{name:<20} {df.shape[0]:>9,} rows x {df.shape[1]:>2} cols")

## Per-dataset shape: dtype, null share, distinct values, an example

In [ ]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    """One row per column: dtype, how populated it is, how varied, and a sample value.

    Blank strings count as empty — several columns in these files use "" rather than
    NA for missing (`location`, `report_date`), so a plain isna() reads as full coverage.

    `n_filled` is a count rather than only a percentage because these files are sparse
    enough for rounding to lie: `security_vintage_year` is 99.993% empty, which prints
    as 100.0 while still carrying 151 real values.
    """
    blank_or_na = df.isna() | df.apply(lambda s: s.astype("string").str.strip().eq(""))
    filled = df.mask(blank_or_na)
    return pd.DataFrame(
        {
            "dtype": df.dtypes.astype("string"),
            "n_filled": filled.notna().sum(),
            "pct_empty": (blank_or_na.mean() * 100).round(1),
            "distinct": filled.nunique(dropna=True),
            "example": [filled[c].dropna().iloc[0] if filled[c].notna().any() else pd.NA for c in df.columns],
        }
    )


profile(company_info)

In [ ]:
profile(corporate_structure)

In [ ]:
profile(shareholders)

## Peek at rows

Widen the display so long free-text columns (`hq_address`, `source_quote`, `text`)
aren't truncated to nothing.

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

company_info.head()

In [ ]:
corporate_structure.head()

In [ ]:
shareholders.head()

## Scratch

One thing worth knowing before you join anything: `corporate_structure.parent_cik` and
`cdt.cik` are both unpadded, and `company_info.identifier` is zero-padded to 10, so the
naive join matches zero rows without failing. `build_dataset.normalize_cik` is the shared
fix.

The CDT files need one more precaution: `cdt_mentions.debt_instrument_mention_id` has 23
duplicate rows and `cdt_items.item_id` has 803, so join through
`.drop_duplicates(key)` or the instruments multiply — 1,640 becomes 1,861.